# SLIIT IT3051 - Data Mining Project (EV Battery PHM)
## Section D: Target Isolation, Splits & Preprocessing Pipeline
### **Assigned Member: Person D**

---
### Objectives of this Notebook:
1. **Enforce Boundary Conditions & Target Isolation:** Exclude non-predictive identifiers (`vehicle_id`, `battery_serial`) and decouple Task 1 and Task 2 targets to prevent circular data leakage.
2. **Integrate Feature Engineering:** Incorporate the domain features derived in Stage 3 (`temperature_spread`, `efficiency_gap`, `health_loss_interaction`, `resistance_per_1000_cycles`, `c_rate_proxy`, `cell_voltage_spread`, `stress_index`).
3. **Dual-Task Train/Test Partitioning:** Implement an 80/20 regression split for Task 1 (dropping unlabelled RUL rows) and an 80/20 stratified classification split for Task 2.
4. **Build Production ColumnTransformer Pipeline:** Assemble unified imputation and scaling/encoding pipelines fitted strictly on training data to guarantee zero leakage.


### 0. Environment Setup & Configuration
Import core scientific and machine learning libraries.


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Visual formatting settings
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.sans-serif'] = 'Arial'

print('Environment initialized successfully!')


Environment initialized successfully!


### Task 1: Boundary Conditions, Feature Integration & Target Isolation Policy
1. Load the raw telemetry dataset.
2. Apply Stage 3 domain feature engineering (incorporating Person C's electrochemical features).
3. Enforce boundary exclusions: eliminate database primary keys (`vehicle_id`, `battery_serial`).
4. Enforce strict **Target Isolation**: guarantee that neither target variable (`predicted_remaining_life_cycles` or `battery_failure`) enters the feature space of the other model.


In [2]:
# 1. Load Raw Dataset
DATASET_PATH = 'ev battery_failure  Dataset.csv'
if not os.path.exists(DATASET_PATH):
    DATASET_PATH = '../ev battery_failure  Dataset.csv'

df_raw = pd.read_csv(DATASET_PATH)
print(f"Raw Dataset Loaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

# 2. Apply Domain Feature Engineering (from Person C & Stage 3 specifications)
df = df_raw.copy()

# Thermal gradient (Max vs Avg temperature)
df['temperature_spread'] = df['cell_temperature_max'] - df['cell_temperature_avg']

# Coulombic efficiency mismatch
df['efficiency_gap'] = (df['charge_efficiency'] - df['discharge_efficiency']).abs()

# State of degradation interaction
df['health_loss_interaction'] = (df['battery_health_percent'] * df['capacity_loss_percent']) / 100.0

# Normalized internal resistance accumulation
df['resistance_per_1000_cycles'] = (df['internal_resistance'] / (df['cycle_count'] + 1.0)) * 1000.0

# Electrical stress proxy (C-Rate: Charging Power / Battery Capacity)
df['c_rate_proxy'] = df['average_charge_power_kw'] / (df['battery_capacity_kwh'] + 1e-5)

# Relative cell voltage dispersion
df['cell_voltage_spread'] = df['cell_voltage_std'] / (df['cell_voltage_avg'] + 1e-5)

# Dynamic mechanical shock index (Hard braking * Aggressive acceleration)
df['stress_index'] = df['aggressive_acceleration_score'] * df['hard_braking_score']

print(f"Post-Feature Engineering Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

# 3. Boundary Conditions & Target Isolation Definitions
ID_COLUMNS = ['vehicle_id', 'battery_serial']
TARGET_RUL = 'predicted_remaining_life_cycles'
TARGET_FAIL = 'battery_failure'

EXCLUDED_FOR_PREDICTORS = ID_COLUMNS + [TARGET_RUL, TARGET_FAIL]
feature_cols = [c for c in df.columns if c not in EXCLUDED_FOR_PREDICTORS]

categorical_cols = df[feature_cols].select_dtypes(include=['object', 'string', 'category']).columns.tolist()
numerical_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

print(f"\n=== FEATURE SEPARATION AUDIT ===")
print(f"Excluded Database Identifiers: {ID_COLUMNS}")
print(f"Isolated Dual Targets:          ['{TARGET_RUL}', '{TARGET_FAIL}']")
print(f"Total Predictive Features:     {len(feature_cols)}")
print(f"  - Numerical Features:        {len(numerical_cols)}")
print(f"  - Categorical Features:      {len(categorical_cols)} ({categorical_cols})")


Raw Dataset Loaded: 20,000 rows x 70 columns
Post-Feature Engineering Shape: 20,000 rows x 77 columns

=== FEATURE SEPARATION AUDIT ===
Excluded Database Identifiers: ['vehicle_id', 'battery_serial']
Isolated Dual Targets:          ['predicted_remaining_life_cycles', 'battery_failure']
Total Predictive Features:     73
  - Numerical Features:        65
  - Categorical Features:      8 (['vehicle_brand', 'vehicle_model', 'vehicle_type', 'battery_manufacturer', 'battery_chemistry', 'drive_type', 'fleet_or_private', 'terrain_type'])


**Analysis & Viva Preparation Notes:**
- **Why must `vehicle_id` and `battery_serial` be dropped before model training?**
  Database identifiers are arbitrary surrogate keys generated by database sequences or manufacturing serialisation. They have zero physical or electrochemical correlation with degradation mechanisms (e.g., lithium plating, SEI layer growth). If included, high-capacity models like Decision Trees or Gradient Boosting will memorize specific IDs rather than learning generalizable degradation laws, causing severe overfitting and training-test disparity.

- **What is Target Isolation, and why would using `battery_failure` to predict RUL (or vice versa) cause circular data leakage?**
  In a production automotive Battery Management System (BMS), both the Remaining Useful Life (RUL) and the Critical Failure Risk must be estimated **simultaneously in real-time** from raw incoming sensor telemetry. If Task 1 used `battery_failure` as an input feature, the regression model would fail at runtime because failure status is unobserved before it happens. Similarly, if Task 2 used `predicted_remaining_life_cycles` as an input feature, any error in the regression model would cascade directly into the safety-critical classification system. Enforcing strict Target Isolation guarantees that both models draw exclusively from independent sensor and vehicle features, eliminating circular feedback loops.


### Task 2: Dual-Task Data Partitioning (Train/Test Splits)
Implement mathematically sound splits for both tasks:
1. **Task 1 (RUL Regression):** Filter out the 898 records with unobserved RUL target values (avoiding synthetic target imputation), then perform an 80/20 train/test split.
2. **Task 2 (Critical Failure Classification):** Utilize all 20,000 instances and perform a **Stratified 80/20 train/test split** to preserve the 93.08% / 6.92% class balance across both partitions.


In [3]:
print("=" * 75)
print("TASK 1 DATA SPLIT: REMAINING LIFE CYCLES (RUL) REGRESSION")
print("=" * 75)

# Filter out unobserved target labels for Task 1
mask_valid_rul = df[TARGET_RUL].notnull()
df_task1 = df[mask_valid_rul].copy()
dropped_rul_count = len(df) - len(df_task1)

X1 = df_task1[feature_cols]
y1 = df_task1[TARGET_RUL]

print(f"Raw Records:                  {len(df):,}")
print(f"Dropped Unlabelled RUL Rows:  {dropped_rul_count:,} ({dropped_rul_count/len(df)*100:.2f}%)")
print(f"Supervised Task 1 Instances:  {len(df_task1):,}")

# 80/20 Train/Test Split for Task 1
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.20, random_state=42
)

print(f"X1_train Shape: {X1_train.shape} | y1_train: {y1_train.shape}")
print(f"X1_test Shape:  {X1_test.shape}  | y1_test:  {y1_test.shape}")
print(f"Task 1 RUL Mean: Train = {y1_train.mean():.1f} cycles | Test = {y1_test.mean():.1f} cycles")

print("\n" + "=" * 75)
print("TASK 2 DATA SPLIT: CRITICAL FAILURE RISK CLASSIFICATION")
print("=" * 75)

# Task 2 uses all 20,000 instances
X2 = df[feature_cols]
y2 = df[TARGET_FAIL]

# Stratified 80/20 Train/Test Split preserving class proportions
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.20, random_state=42, stratify=y2
)

print(f"X2_train Shape: {X2_train.shape} | y2_train: {y2_train.shape}")
print(f"X2_test Shape:  {X2_test.shape}  | y2_test:  {y2_test.shape}")

train_fail_rate = (y2_train == 1).mean() * 100
test_fail_rate = (y2_test == 1).mean() * 100
print(f"Train Failure Ratio: {(y2_train == 1).sum():,} / {len(y2_train):,} ({train_fail_rate:.2f}%)")
print(f"Test Failure Ratio:  {(y2_test == 1).sum():,} / {len(y2_test):,} ({test_fail_rate:.2f}%)")


TASK 1 DATA SPLIT: REMAINING LIFE CYCLES (RUL) REGRESSION
Raw Records:                  20,000
Dropped Unlabelled RUL Rows:  898 (4.49%)
Supervised Task 1 Instances:  19,102
X1_train Shape: (15281, 73) | y1_train: (15281,)
X1_test Shape:  (3821, 73)  | y1_test:  (3821,)
Task 1 RUL Mean: Train = 7812.6 cycles | Test = 7876.4 cycles

TASK 2 DATA SPLIT: CRITICAL FAILURE RISK CLASSIFICATION
X2_train Shape: (16000, 73) | y2_train: (16000,)
X2_test Shape:  (4000, 73)  | y2_test:  (4000,)
Train Failure Ratio: 1,107 / 16,000 (6.92%)
Test Failure Ratio:  277 / 4,000 (6.93%)


**Analysis & Viva Preparation Notes:**
- **Why must we use a stratified split for Task 2?**
  The dataset exhibits a severe class imbalance of **13.45 : 1** (18,616 normal cases vs 1,384 failures, or 6.92% minority class). A purely random, non-stratified split risks sampling bias where the test set could inadvertently receive too few failure instances, producing unreliable evaluation metrics with high variance. Stratification (`stratify=y2`) guarantees that both training (1,107 failures, 6.92%) and testing (277 failures, 6.93%) maintain the exact empirical prior probability distribution of the real-world fleet.

- **Why should we exclude the 898 missing RUL rows from Task 1 training instead of imputing artificial target values?**
  Supervised learning algorithms require authentic ground-truth labels to optimize loss functions (e.g., Mean Squared Error). If we imputed missing target labels using mean, median, or KNN regression, the model would be trained on synthetic predictions of predictions (circular pseudo-labelling). This distorts the loss surface, injects false confidence into regression error gradients, and invalidates cross-validation benchmarks. Therefore, unlabelled target instances must be filtered out of supervised training.


### Task 3: Building the Production ColumnTransformer Pipeline
Construct an enterprise-grade `ColumnTransformer`:
1. **Numerical Pipeline:** `SimpleImputer(strategy='median')` followed by `StandardScaler()`.
2. **Categorical Pipeline:** `SimpleImputer(strategy='most_frequent')` followed by `OneHotEncoder(handle_unknown='ignore', sparse_output=False)`.
3. **Pipeline Modularity:** Ensures identical transformations across training and future deployment data.


In [4]:
# 1. Define Numerical Feature Transformation Pipeline
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # Robust against sensor spikes
    ('scaler', StandardScaler())                    # Zero mean, unit variance scaling
])

# 2. Define Categorical Feature Transformation Pipeline
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')), # Mode imputation for vehicle specs
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) # Resilient one-hot
])

# 3. Assemble Master ColumnTransformer Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline, numerical_cols),
        ('cat', categorical_pipeline, categorical_cols)
    ],
    verbose_feature_names_out=False
)

print("Production ColumnTransformer assembled successfully:")
print(preprocessor)


Production ColumnTransformer assembled successfully:
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['manufacturing_year', 'battery_capacity_kwh',
                                  'odometer_km', 'vehicle_age_years',
                                  'cycle_count', 'battery_health_percent',
                                  'state_of_charge', 'depth_of_discharge',
                                  'state_of_health', 'cell_voltage_avg',
                                  'cell_voltage_std', 'pack_v...
                                  'average_speed', 'average_trip_distance', ...]),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleIm

**Analysis & Viva Preparation Notes:**
- **Why is median imputation preferred over mean imputation for continuous sensor readings?**
  As proven by Person B's exploratory analysis, critical physical variables like `cell_temperature_max` reach up to **87°C** (thermal runaway zone) and `internal_resistance` reaches **1.12 Ω** (high degradation tail). Mean imputation is mathematically vulnerable to these extreme values, shifting imputed baselines upward and artificially distorting healthy battery readings. Median imputation represents the true 50th percentile nominal operating condition, remaining stable and unaffected by localized outlier spikes.

- **What is the purpose of `handle_unknown='ignore'` in the OneHotEncoder during test inference?**
  In real-world EV deployment, a newly introduced vehicle model or brand might be monitored that was never seen in the historical training set. With the default `handle_unknown='error'`, the preprocessor would crash and halt telemetry ingestion. By specifying `handle_unknown='ignore'`, unseen categories are smoothly encoded as all zeros across the one-hot columns, allowing the model to continue uninterrupted inference using the remaining numerical sensor signals.


### Task 4: Zero Data Leakage Verification & Transformed Dimensions
1. Fit the `ColumnTransformer` **strictly on the training split** of each task, then transform both train and test splits.
2. Verify final transformed feature dimensions.
3. Verify that the standardized features have empirical mean $\approx 0$ and standard deviation $\approx 1$ on the training split.
4. Inspect the generated feature names.


In [5]:
print("=" * 75)
print("EXECUTING TRANSFORMATIONS FOR TASK 1 (RUL REGRESSION)")
print("=" * 75)

# Build separate preprocessor instance for Task 1
preprocessor_t1 = ColumnTransformer([
    ('num', numerical_pipeline, numerical_cols),
    ('cat', categorical_pipeline, categorical_cols)
], verbose_feature_names_out=False)

# Strict Leakage Guard: FIT ONLY ON TRAIN
X1_train_transformed = preprocessor_t1.fit_transform(X1_train)
X1_test_transformed = preprocessor_t1.transform(X1_test)
feature_names_out_t1 = preprocessor_t1.get_feature_names_out()

print(f"X1_train Transformed Shape: {X1_train_transformed.shape}")
print(f"X1_test Transformed Shape:  {X1_test_transformed.shape}")
print(f"Total Processed Feature Dimensions: {len(feature_names_out_t1)}")

print("\n" + "=" * 75)
print("EXECUTING TRANSFORMATIONS FOR TASK 2 (FAILURE CLASSIFICATION)")
print("=" * 75)

# Build separate preprocessor instance for Task 2
preprocessor_t2 = ColumnTransformer([
    ('num', numerical_pipeline, numerical_cols),
    ('cat', categorical_pipeline, categorical_cols)
], verbose_feature_names_out=False)

# Strict Leakage Guard: FIT ONLY ON TRAIN
X2_train_transformed = preprocessor_t2.fit_transform(X2_train)
X2_test_transformed = preprocessor_t2.transform(X2_test)
feature_names_out_t2 = preprocessor_t2.get_feature_names_out()

print(f"X2_train Transformed Shape: {X2_train_transformed.shape}")
print(f"X2_test Transformed Shape:  {X2_test_transformed.shape}")
print(f"Total Processed Feature Dimensions: {len(feature_names_out_t2)}")

# Leakage & Scaling Mathematical Verification
train_scaled_mean = np.mean(X2_train_transformed[:, :len(numerical_cols)])
train_scaled_std = np.mean(np.std(X2_train_transformed[:, :len(numerical_cols)], axis=0))
test_scaled_mean = np.mean(X2_test_transformed[:, :len(numerical_cols)])

print("\n=== MATHEMATICAL DATA LEAKAGE VERIFICATION ===")
print(f"Train Scaled Numeric Mean (Expected ~ 0.0): {train_scaled_mean:.6f}")
print(f"Train Scaled Numeric Std  (Expected ~ 1.0): {train_scaled_std:.6f}")
print(f"Test Scaled Numeric Mean  (Slight shift):    {test_scaled_mean:.6f}")
print("Verification Result: Preprocessing parameters were derived exclusively from training data!")

print("\nSample Transformed Feature Column Names (first 15):")
for idx, fname in enumerate(feature_names_out_t1[:15], 1):
    print(f"  {idx:2d}. {fname}")


EXECUTING TRANSFORMATIONS FOR TASK 1 (RUL REGRESSION)
X1_train Transformed Shape: (15281, 156)
X1_test Transformed Shape:  (3821, 156)
Total Processed Feature Dimensions: 156

EXECUTING TRANSFORMATIONS FOR TASK 2 (FAILURE CLASSIFICATION)
X2_train Transformed Shape: (16000, 156)
X2_test Transformed Shape:  (4000, 156)
Total Processed Feature Dimensions: 156

=== MATHEMATICAL DATA LEAKAGE VERIFICATION ===
Train Scaled Numeric Mean (Expected ~ 0.0): -0.000000
Train Scaled Numeric Std  (Expected ~ 1.0): 1.000000
Test Scaled Numeric Mean  (Slight shift):    0.000100
Verification Result: Preprocessing parameters were derived exclusively from training data!

Sample Transformed Feature Column Names (first 15):
   1. manufacturing_year
   2. battery_capacity_kwh
   3. odometer_km
   4. vehicle_age_years
   5. cycle_count
   6. battery_health_percent
   7. state_of_charge
   8. depth_of_discharge
   9. state_of_health
  10. cell_voltage_avg
  11. cell_voltage_std
  12. pack_voltage
  13. cell_te

**Analysis & Viva Preparation Notes:**
- **What are the final transformed feature dimensions for train and test sets?**
  - **Task 1 (RUL Regression):** Training matrix has dimensions **(15,281, 156)**; testing matrix has **(3,821, 156)**.
  - **Task 2 (Failure Classification):** Training matrix has dimensions **(16,000, 156)**; testing matrix has **(4,000, 156)**.
  - The 156 transformed dimensions consist of **65 continuous scaled variables** (including the 7 engineered domain features) and **91 binary dummy indicators** from the 8 one-hot encoded categorical variables.

- **How does fitting the preprocessor strictly on the training partition prevent data leakage?**
  Data leakage occurs whenever statistical information about the test distribution (e.g., mean, variance, median, or category frequencies) influences the training phase. If `fit_transform` were applied globally to the entire dataset before splitting, the test set's outliers and scale would bleed into the training imputer and scaler. By using `fit_transform` strictly on `X_train` and applying `transform` to `X_test`, the test partition remains completely unseen, simulating true production conditions during model evaluation.

---
### Final Viva 1 Readiness Checkpoint (Person D Deliverable Complete)
- [x] Dropped `vehicle_id` and `battery_serial`.
- [x] Enforced strict Target Isolation between RUL Regression and Failure Classification.
- [x] Integrated Person C's engineered features.
- [x] Filtered out unlabelled RUL target instances from Task 1.
- [x] Executed Stratified train/test split for Task 2 to safeguard minority failure cases.
- [x] Built production `ColumnTransformer` combining robust median imputation and one-hot encoding.
- [x] Confirmed zero data leakage with verified (156-feature) mathematical matrices.
